In [ ]:
# %% [markdown]
# # Análisis Descriptivo PISA 2022
# ## Preparación de datos para LCA e IRT
# 
# Este notebook cubre:
# 1. Carga y fusión de bases de datos PISA
# 2. Análisis descriptivo completo
# 3. Correlaciones y visualizaciones
# 4. Preparación para modelos LCA e IRT

# %% [markdown]
# ### 1. Instalación y carga de librerías

# %%
# Instalar librerías necesarias (ejecutar solo una vez)
# !pip install pandas numpy matplotlib seaborn scipy pyreadstat

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pyreadstat  # Para leer archivos .sav de SPSS
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualizaciones
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✓ Librerías cargadas exitosamente")

✓ Librerías cargadas exitosamente


In [7]:
# %% [markdown]
# ### 2. Carga de bases de datos PISA

# %%
# IMPORTANTE: Ajusta las rutas a donde tienes tus archivos
ruta_base = "../data_ready/COLLECT_COMP_dataset_treated.csv"  # Cambia esta ruta

# Diccionario con las bases de datos
archivos = {
    'escuela': 'CY08MSP_SCH_QQQ.sav',
    'docentes': 'CY08MSP_TCH_QQQ.sav',
    'cognitivo_crt': 'CY08MSP_CRT_COG.sav',
    'cognitivo_stu': 'CY08MSP_STU_COG.sav',
    'estudiantes': 'CY08MSP_STU_QQQ.sav',
    'tiempo': 'CY08MSP_STU_TIM.sav'
}

# Función para cargar datos
def cargar_pisa(nombre_archivo):
    try:
        df, meta = pyreadstat.read_sav(ruta_base + nombre_archivo)
        print(f"✓ {nombre_archivo}: {df.shape[0]} filas, {df.shape[1]} columnas")
        return df, meta
    except FileNotFoundError:
        print(f"✗ Archivo no encontrado: {nombre_archivo}")
        return None, None

# Cargar todas las bases
datos = {}
for key, archivo in archivos.items():
    df, meta = cargar_pisa(archivo)
    if df is not None:
        datos[key] = df

PyreadstatError: File ../data_ready/COLLECT_COMP_dataset_treated.csvCY08MSP_SCH_QQQ.sav does not exist!

In [ ]:
# %% [markdown]
# ### 3. Inspección inicial de datos

# %%
# Verificar claves de unión disponibles
if 'estudiantes' in datos:
    print("Columnas de identificación en estudiantes:")
    cols_id = [col for col in datos['estudiantes'].columns if any(x in col.upper() 
               for x in ['CNT', 'ID', 'SCHID', 'STUID'])]
    print(cols_id[:10])  # Mostrar primeras 10
    
    print(f"\nDimensiones: {datos['estudiantes'].shape}")
    print(f"\nPrimeras columnas: {list(datos['estudiantes'].columns[:15])}")

In [ ]:
# %% [markdown]
# ### 4. Fusión de bases de datos

# %%
# Unir bases de datos a nivel estudiante
# IMPORTANTE: Ajusta las claves según tu dataset real

if 'estudiantes' in datos and 'cognitivo_stu' in datos:
    # Identificar claves comunes
    claves_estudiante = ['CNTRYID', 'CNTSCHID', 'CNTSTUID']
    
    # Fusionar datos cognitivos
    df_completo = datos['estudiantes'].merge(
        datos['cognitivo_stu'],
        on=claves_estudiante,
        how='left',
        suffixes=('', '_cog')
    )
    
    # Fusionar datos de tiempo si existen
    if 'tiempo' in datos:
        df_completo = df_completo.merge(
            datos['tiempo'],
            on=claves_estudiante,
            how='left',
            suffixes=('', '_time')
        )
    
    print(f"✓ Dataset unificado: {df_completo.shape}")
    print(f"  - Filas: {df_completo.shape[0]:,}")
    print(f"  - Columnas: {df_completo.shape[1]:,}")
else:
    print("✗ No se pudieron cargar los datos base")

In [ ]:
# %% [markdown]
# ### 5. Selección de variables de interés

# %%
# Definir variables clave para el análisis
# NOTA: Ajusta según las variables disponibles en tu dataset

variables_interes = {
    'ids': ['CNTRYID', 'CNTSCHID', 'CNTSTUID'],
    'pesos': ['W_FSTUWT'],  # Peso muestral principal
    'rendimiento': [
        'PV1MATH', 'PV2MATH', 'PV3MATH', 'PV4MATH', 'PV5MATH',  # Matemáticas
        'PV1READ', 'PV2READ', 'PV3READ', 'PV4READ', 'PV5READ',  # Lectura
        'PV1SCIE', 'PV2SCIE', 'PV3SCIE', 'PV4SCIE', 'PV5SCIE',  # Ciencias
    ],
    'contexto': [
        'ST004D01T',  # Género
        'ESCS',       # Índice socioeconómico
        'HOMEPOS',    # Recursos del hogar
        'WEALTH',     # Riqueza familiar
        'CULTPOSS',   # Posesiones culturales
        'HEDRES',     # Recursos educativos del hogar
    ]
}

# Verificar qué variables existen
variables_disponibles = []
for categoria, vars_list in variables_interes.items():
    for var in vars_list:
        if var in df_completo.columns:
            variables_disponibles.append(var)

print(f"Variables disponibles: {len(variables_disponibles)}")
print("\nPrimeras 10 variables:")
for var in variables_disponibles[:10]:
    print(f"  - {var}")

# Crear dataset de trabajo
df_trabajo = df_completo[variables_disponibles].copy()
print(f"\n✓ Dataset de trabajo creado: {df_trabajo.shape}")

In [ ]:
# %% [markdown]
# ### 6. Análisis de datos faltantes

# %%
# Porcentaje de datos faltantes por variable
missing = pd.DataFrame({
    'Variable': df_trabajo.columns,
    'Missing_n': df_trabajo.isnull().sum(),
    'Missing_%': (df_trabajo.isnull().sum() / len(df_trabajo) * 100).round(2)
}).sort_values('Missing_%', ascending=False)

print("Top 10 variables con más datos faltantes:")
print(missing.head(10))

# Visualización
plt.figure(figsize=(12, 6))
top_missing = missing.head(20)
plt.barh(top_missing['Variable'], top_missing['Missing_%'], color='coral')
plt.xlabel('Porcentaje de datos faltantes')
plt.title('Variables con más datos faltantes (Top 20)')
plt.tight_layout()
plt.show()

In [ ]:
# %% [markdown]
# ### 7. Estadísticas descriptivas

# %%
# Función para calcular estadísticas descriptivas completas
def estadisticas_descriptivas(df, variables):
    """
    Calcula estadísticas descriptivas para variables continuas
    """
    stats_dict = {
        'Variable': [],
        'N': [],
        'Media': [],
        'DE': [],
        'Min': [],
        'Q1': [],
        'Mediana': [],
        'Q3': [],
        'Max': [],
        'Asimetría': [],
        'Curtosis': []
    }
    
    for var in variables:
        if var in df.columns and df[var].dtype in ['float64', 'int64']:
            data = df[var].dropna()
            if len(data) > 0:
                stats_dict['Variable'].append(var)
                stats_dict['N'].append(len(data))
                stats_dict['Media'].append(data.mean())
                stats_dict['DE'].append(data.std())
                stats_dict['Min'].append(data.min())
                stats_dict['Q1'].append(data.quantile(0.25))
                stats_dict['Mediana'].append(data.median())
                stats_dict['Q3'].append(data.quantile(0.75))
                stats_dict['Max'].append(data.max())
                stats_dict['Asimetría'].append(data.skew())
                stats_dict['Curtosis'].append(data.kurtosis())
    
    return pd.DataFrame(stats_dict)

# Calcular para variables de rendimiento
vars_rendimiento = [v for v in variables_disponibles if 'PV1' in v]
vars_contexto = [v for v in variables_disponibles if v in ['ESCS', 'HOMEPOS', 'WEALTH']]

print("ESTADÍSTICAS DE RENDIMIENTO ACADÉMICO")
print("=" * 80)
stats_rendimiento = estadisticas_descriptivas(df_trabajo, vars_rendimiento)
print(stats_rendimiento.round(2))

print("\n\nESTADÍSTICAS DE VARIABLES DE CONTEXTO")
print("=" * 80)
stats_contexto = estadisticas_descriptivas(df_trabajo, vars_contexto)
print(stats_contexto.round(2))

In [ ]:
# %% [markdown]
# ### 8. Trabajando con Plausible Values

# %%
# IMPORTANTE: PISA usa 10 plausible values, no solo uno
# Debes analizar con todos y combinar resultados

# Calcular media de cada estudiante a través de los 5 PV (ejemplo con matemáticas)
pv_math_cols = [col for col in df_trabajo.columns if 'PV' in col and 'MATH' in col]
if pv_math_cols:
    df_trabajo['MATH_mean'] = df_trabajo[pv_math_cols].mean(axis=1)
    df_trabajo['MATH_sd'] = df_trabajo[pv_math_cols].std(axis=1)
    
    print("Promedio de Plausible Values - Matemáticas:")
    print(f"  Media: {df_trabajo['MATH_mean'].mean():.2f}")
    print(f"  DE: {df_trabajo['MATH_mean'].std():.2f}")
    print(f"  Variabilidad intra-estudiante promedio: {df_trabajo['MATH_sd'].mean():.2f}")

# Mismo proceso para lectura y ciencias
pv_read_cols = [col for col in df_trabajo.columns if 'PV' in col and 'READ' in col]
if pv_read_cols:
    df_trabajo['READ_mean'] = df_trabajo[pv_read_cols].mean(axis=1)

pv_scie_cols = [col for col in df_trabajo.columns if 'PV' in col and 'SCIE' in col]
if pv_scie_cols:
    df_trabajo['SCIE_mean'] = df_trabajo[pv_scie_cols].mean(axis=1)

In [ ]:
# %% [markdown]
# ### 9. Distribuciones de variables clave

# %%
# Crear figura con múltiples subplots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Distribuciones de Variables Clave', fontsize=16, fontweight='bold')

# Variables a visualizar
vars_plot = ['MATH_mean', 'READ_mean', 'SCIE_mean', 'ESCS', 'HOMEPOS', 'WEALTH']
titles = ['Matemáticas', 'Lectura', 'Ciencias', 'ESCS (NSE)', 'Recursos Hogar', 'Riqueza']

for idx, (var, title) in enumerate(zip(vars_plot, titles)):
    ax = axes[idx // 3, idx % 3]
    if var in df_trabajo.columns:
        data = df_trabajo[var].dropna()
        
        # Histograma
        ax.hist(data, bins=50, alpha=0.7, edgecolor='black', color='steelblue')
        ax.axvline(data.mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {data.mean():.1f}')
        ax.axvline(data.median(), color='green', linestyle='--', linewidth=2, label=f'Mediana: {data.median():.1f}')
        
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Valor')
        ax.set_ylabel('Frecuencia')
        ax.legend()
        ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'Variable no disponible', ha='center', va='center')
        ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
# %% [markdown]
# ### 10. Matriz de correlaciones

# %%
# Seleccionar variables continuas para correlación
vars_corr = ['MATH_mean', 'READ_mean', 'SCIE_mean', 'ESCS', 'HOMEPOS', 'WEALTH', 'CULTPOSS', 'HEDRES']
vars_corr_disponibles = [v for v in vars_corr if v in df_trabajo.columns]

if len(vars_corr_disponibles) > 1:
    # Calcular matriz de correlación
    corr_matrix = df_trabajo[vars_corr_disponibles].corr()
    
    # Visualización con mapa de calor
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
                square=True, linewidths=1, cbar_kws={"shrink": 0.8})
    plt.title('Matriz de Correlaciones (Pearson)', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    
    # Tabla de correlaciones más fuertes
    print("\nCorrelaciones más fuertes (|r| > 0.5):")
    print("=" * 60)
    corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            corr_val = corr_matrix.iloc[i, j]
            if abs(corr_val) > 0.5:
                corr_pairs.append({
                    'Variable 1': corr_matrix.columns[i],
                    'Variable 2': corr_matrix.columns[j],
                    'Correlación': corr_val
                })
    
    if corr_pairs:
        df_corr = pd.DataFrame(corr_pairs).sort_values('Correlación', ascending=False, key=abs)
        print(df_corr.to_string(index=False))
    else:
        print("No hay correlaciones mayores a 0.5")

In [ ]:
# %% [markdown]
# ### 11. Gráficos de dispersión clave

# %%
# Relación entre NSE y rendimiento académico
if 'ESCS' in df_trabajo.columns and 'MATH_mean' in df_trabajo.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    subjects = [('MATH_mean', 'Matemáticas'), 
                ('READ_mean', 'Lectura'), 
                ('SCIE_mean', 'Ciencias')]
    
    for idx, (var, title) in enumerate(subjects):
        if var in df_trabajo.columns:
            # Scatter plot con línea de tendencia
            ax = axes[idx]
            
            # Muestra aleatoria para mejor visualización
            sample = df_trabajo[['ESCS', var]].dropna().sample(min(5000, len(df_trabajo)))
            
            ax.scatter(sample['ESCS'], sample[var], alpha=0.3, s=10, color='steelblue')
            
            # Línea de tendencia
            z = np.polyfit(sample['ESCS'], sample[var], 1)
            p = np.poly1d(z)
            ax.plot(sample['ESCS'].sort_values(), p(sample['ESCS'].sort_values()), 
                   "r-", linewidth=2, label=f'y = {z[0]:.1f}x + {z[1]:.1f}')
            
            # Calcular correlación
            corr = sample['ESCS'].corr(sample[var])
            
            ax.set_xlabel('ESCS (Índice Socioeconómico)', fontsize=11)
            ax.set_ylabel(f'Puntaje {title}', fontsize=11)
            ax.set_title(f'{title} vs NSE (r = {corr:.3f})', fontweight='bold')
            ax.legend()
            ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# %% [markdown]
# ### 12. Análisis por grupos

# %%
# Análisis por género (si está disponible)
if 'ST004D01T' in df_trabajo.columns:
    print("ANÁLISIS POR GÉNERO")
    print("=" * 80)
    
    # Recodificar género (ajusta según tu codificación)
    df_trabajo['Genero'] = df_trabajo['ST004D01T'].map({1: 'Mujer', 2: 'Hombre'})
    
    # Estadísticas por género
    for var in ['MATH_mean', 'READ_mean', 'SCIE_mean']:
        if var in df_trabajo.columns:
            print(f"\n{var}:")
            print(df_trabajo.groupby('Genero')[var].describe().round(2))
    
    # Visualización
    if all(v in df_trabajo.columns for v in ['MATH_mean', 'READ_mean', 'SCIE_mean']):
        df_plot = df_trabajo[['Genero', 'MATH_mean', 'READ_mean', 'SCIE_mean']].dropna()
        
        fig, ax = plt.subplots(figsize=(10, 6))
        df_plot.groupby('Genero')[['MATH_mean', 'READ_mean', 'SCIE_mean']].mean().plot(
            kind='bar', ax=ax, width=0.7, color=['#FF6B6B', '#4ECDC4', '#45B7D1']
        )
        ax.set_ylabel('Puntaje Promedio', fontsize=12)
        ax.set_xlabel('Género', fontsize=12)
        ax.set_title('Rendimiento Académico por Género', fontsize=14, fontweight='bold')
        ax.legend(['Matemáticas', 'Lectura', 'Ciencias'])
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

In [ ]:
# %% [markdown]
# ### 13. Análisis por cuartiles socioeconómicos

# %%
if 'ESCS' in df_trabajo.columns:
    # Crear cuartiles
    df_trabajo['ESCS_quartil'] = pd.qcut(df_trabajo['ESCS'], q=4, 
                                          labels=['Q1 (Bajo)', 'Q2', 'Q3', 'Q4 (Alto)'])
    
    print("ANÁLISIS POR CUARTIL SOCIOECONÓMICO")
    print("=" * 80)
    
    # Tamaño de cada cuartil
    print("\nDistribución por cuartil:")
    print(df_trabajo['ESCS_quartil'].value_counts().sort_index())
    
    # Estadísticas por cuartil
    for var in ['MATH_mean', 'READ_mean', 'SCIE_mean']:
        if var in df_trabajo.columns:
            print(f"\n{var} por cuartil:")
            print(df_trabajo.groupby('ESCS_quartil')[var].describe()[['mean', 'std']].round(2))
    
    # Visualización
    if 'MATH_mean' in df_trabajo.columns:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        # Gráfico de barras
        df_plot = df_trabajo.groupby('ESCS_quartil')[['MATH_mean', 'READ_mean', 'SCIE_mean']].mean()
        df_plot.plot(kind='bar', ax=axes[0], width=0.8, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
        axes[0].set_ylabel('Puntaje Promedio', fontsize=12)
        axes[0].set_xlabel('Cuartil Socioeconómico', fontsize=12)
        axes[0].set_title('Rendimiento por Nivel Socioeconómico', fontsize=13, fontweight='bold')
        axes[0].legend(['Matemáticas', 'Lectura', 'Ciencias'])
        axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
        axes[0].grid(axis='y', alpha=0.3)
        
        # Box plot
        df_trabajo.boxplot(column='MATH_mean', by='ESCS_quartil', ax=axes[1])
        axes[1].set_ylabel('Puntaje Matemáticas', fontsize=12)
        axes[1].set_xlabel('Cuartil Socioeconómico', fontsize=12)
        axes[1].set_title('Distribución de Puntajes en Matemáticas', fontsize=13, fontweight='bold')
        plt.suptitle('')  # Remover título automático
        
        plt.tight_layout()
        plt.show()

In [ ]:
# %% [markdown]
# ### 14. Identificación de valores atípicos

# %%
def detectar_outliers(df, columna, metodo='iqr'):
    """
    Detecta outliers usando el método IQR
    """
    if columna not in df.columns:
        return None
    
    data = df[columna].dropna()
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    
    outliers = (data < limite_inf) | (data > limite_sup)
    
    return {
        'columna': columna,
        'n_outliers': outliers.sum(),
        'pct_outliers': (outliers.sum() / len(data) * 100),
        'limite_inf': limite_inf,
        'limite_sup': limite_sup
    }

print("DETECCIÓN DE VALORES ATÍPICOS")
print("=" * 80)

vars_revisar = ['MATH_mean', 'READ_mean', 'SCIE_mean', 'ESCS']
outliers_info = []

for var in vars_revisar:
    if var in df_trabajo.columns:
        info = detectar_outliers(df_trabajo, var)
        if info:
            outliers_info.append(info)
            print(f"\n{var}:")
            print(f"  - Outliers: {info['n_outliers']} ({info['pct_outliers']:.2f}%)")
            print(f"  - Rango esperado: [{info['limite_inf']:.2f}, {info['limite_sup']:.2f}]")

In [ ]:
# %% [markdown]
# ### 15. Resumen ejecutivo

# %%
print("\n" + "="*80)
print("RESUMEN EJECUTIVO DEL ANÁLISIS DESCRIPTIVO")
print("="*80)

print(f"\n1. TAMAÑO DE LA MUESTRA:")
print(f"   - Total de estudiantes: {len(df_trabajo):,}")
print(f"   - Variables disponibles: {len(df_trabajo.columns)}")

if 'MATH_mean' in df_trabajo.columns:
    print(f"\n2. RENDIMIENTO ACADÉMICO PROMEDIO:")
    print(f"   - Matemáticas: {df_trabajo['MATH_mean'].mean():.1f} (DE: {df_trabajo['MATH_mean'].std():.1f})")
    if 'READ_mean' in df_trabajo.columns:
        print(f"   - Lectura: {df_trabajo['READ_mean'].mean():.1f} (DE: {df_trabajo['READ_mean'].std():.1f})")
    if 'SCIE_mean' in df_trabajo.columns:
        print(f"   - Ciencias: {df_trabajo['SCIE_mean'].mean():.1f} (DE: {df_trabajo['SCIE_mean'].std():.1f})")

if 'ESCS' in df_trabajo.columns and 'MATH_mean' in df_trabajo.columns:
    corr_escs_math = df_trabajo['ESCS'].corr(df_trabajo['MATH_mean'])
    print(f"\n3. CORRELACIONES CLAVE:")
    print(f"   - ESCS ↔ Matemáticas: {corr_escs_math:.3f}")

print(f"\n4. DATOS FALTANTES:")
print(f"   - Variable con más faltantes: {missing.iloc[0]['Variable']} ({missing.iloc[0]['Missing_%']:.1f}%)")

print("\n" + "="*80)
print("✓ Análisis descriptivo completado")
print("="*80)

# %% [markdown]
# ###